# 06 - Treino do Modelo com MLflow

No notebook 05 eu preparei a `gold_ml_features`, com as colunas que o modelo vai usar (medias moveis, volatilidade, retornos passados) e o target (`target_retorno_prox_dia`). Agora vem a parte que estava faltando: treinar um modelo de verdade e acompanhar isso de forma organizada com o **MLflow**.

**Por que MLflow?** Quando voce vai testando parametros diferentes (quantas arvores, profundidade, etc.), e facil perder o controle de qual versao deu o resultado melhor. O MLflow registra automaticamente cada tentativa (chamada de "run"): os parametros usados, as metricas que deram, e ate o modelo treinado, tudo versionado. No Databricks isso ja vem pronto, nao precisa configurar servidor nem nada.

**Por que saio do Spark aqui?** A `gold_ml_features` tem menos de 9 mil linhas -- pequena pra qualquer computador rodar sem problema. scikit-learn e XGBoost (as ferramentas que vou usar pra treinar o modelo) trabalham com pandas/numpy, entao faz mais sentido converter pra pandas aqui do que manter a complexidade do Spark pra um dataset desse tamanho.

O plano deste notebook:
1. Ler a `gold_ml_features` e converter pra pandas
2. Separar treino e teste respeitando a ordem do tempo (nada de embaralhar)
3. Treinar um modelo XGBoost pra prever o retorno do dia seguinte
4. Acompanhar tudo com MLflow (parametros, metricas, modelo)
5. Olhar quais features o modelo considerou mais importantes

In [0]:
%pip install mlflow xgboost scikit-learn

In [0]:
dbutils.library.restartPython()

## Passo 1 -- Lendo a feature store

O `restartPython()` limpa a memoria do notebook (por isso os pacotes novos entram em vigor), entao preciso ler os dados de novo aqui.

`.toPandas()` converte um DataFrame do Spark pra um DataFrame do pandas. So faco isso porque o dataset e pequeno -- se fossem milhoes de linhas, essa conversao ia estourar a memoria e eu precisaria treinar com Spark ML em vez disso.

In [0]:
df_pd = spark.table("b3_pipeline.gold_ml_features").toPandas()
df_pd = df_pd.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"Total de linhas: {len(df_pd)}")
df_pd.head()

## Passo 2 -- Separando treino e teste (respeitando o tempo)

Aqui tem uma pegadinha importante que eu quase caí: com dados normais, a gente costuma usar `train_test_split` embaralhando as linhas aleatoriamente. Com serie temporal isso e um erro grave, porque o modelo acabaria treinando com dados de datas futuras e sendo testado com datas passadas -- ele "veria o futuro" durante o treino, o que se chama de **vazamento de dados (data leakage)**. O resultado pareceria otimo no teste, mas seria inutil na vida real, porque no mundo real eu nunca tenho acesso ao futuro quando preciso prever.

Por isso separo por data: pego uma data de corte, tudo antes dela vira treino, tudo depois vira teste -- exatamente como seria usar o modelo de verdade (so com o passado, prevendo o que ainda nao aconteceu).

In [0]:
# Pego todas as datas unicas em ordem, e escolho a que fica na posicao de 80% da lista
datas_unicas = sorted(df_pd["date"].unique())
data_corte = datas_unicas[int(len(datas_unicas) * 0.8)]

treino = df_pd[df_pd["date"] <= data_corte]
teste = df_pd[df_pd["date"] > data_corte]

print(f"Treino: {len(treino)} linhas | Teste: {len(teste)} linhas")
print(f"Data de corte: {data_corte}")

## Passo 3 -- Escolhendo features (X) e target (y)

Por convencao em ML, `X` e o conjunto de colunas de entrada (features) e `y` e a coluna que quero prever (target). Uso o retorno de hoje (`daily_return_pct`) tambem como feature -- isso nao e vazamento, porque no momento de prever o retorno de amanha eu ja sei qual foi o retorno de hoje.

Deixei de fora `ticker`, `setor`, `date` e `close` por enquanto: sao texto ou data, e um modelo de regressao simples como esse precisa de numeros. Da pra incluir `ticker`/`setor` depois transformando em numero (encoding), mas pra o primeiro modelo prefiro manter simples.

In [0]:
features = [
    "daily_return_pct",
    "media_movel_5d", "media_movel_10d", "volatilidade_5d",
    "retorno_lag1", "retorno_lag3", "retorno_lag5"
]
target = "target_retorno_prox_dia"

X_treino, y_treino = treino[features], treino[target]
X_teste, y_teste = teste[features], teste[target]

## Passo 4 -- Treinando o modelo com MLflow

Uso o **XGBoost**, que e um modelo de *gradient boosting*: ele monta varias arvores de decisao pequenas em sequencia, e cada arvore nova tenta corrigir o erro que as anteriores deixaram passar. E um dos modelos mais usados hoje pra dados em tabela (como o nosso).

`with mlflow.start_run():` abre um "run" -- tudo que acontece dentro desse bloco (parametros, metricas, modelo) fica registrado junto, como uma ficha unica daquela tentativa. Se eu rodar essa celula de novo com outros parametros, vira um novo run, e da pra comparar os dois depois na aba Experiments do Databricks.

Duas metricas pra avaliar o resultado:
- **MAE** (erro absoluto medio): em media, o quanto a previsao errou, em pontos percentuais de retorno. Quanto menor, melhor.
- **R2**: o quanto o modelo explica da variacao do retorno, de 0 a 1 (pode ser negativo se o modelo for pior que simplesmente chutar a media). Adianto: pra retorno de acoes, um R2 baixo (perto de 0) e o resultado mais comum e esperado -- o mercado e extremamente ruidoso, e nenhum modelo simples bate isso com folga. O valor aqui esta em montar o fluxo completo (features, treino, avaliacao, tracking), nao em "vencer a bolsa".

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

with mlflow.start_run(run_name="xgboost_retorno_prox_dia"):
    parametros = {
        "n_estimators": 100,
        "max_depth": 3,
        "learning_rate": 0.05,
        "random_state": 42
    }

    modelo = XGBRegressor(**parametros)
    modelo.fit(X_treino, y_treino)

    y_pred = modelo.predict(X_teste)
    mae = mean_absolute_error(y_teste, y_pred)
    r2 = r2_score(y_teste, y_pred)

    # mlflow.log_param guarda cada configuracao usada no treino
    for nome, valor in parametros.items():
        mlflow.log_param(nome, valor)

    # mlflow.log_metric guarda o resultado da avaliacao
    mlflow.log_metric("mae", mae)
    mlflow.log_metric("r2", r2)

    # mlflow.xgboost.log_model salva o modelo treinado, versionado junto com esse run
    mlflow.xgboost.log_model(modelo, "modelo")

    print(f"MAE: {mae:.4f}")
    print(f"R2: {r2:.4f}")

## Passo 5 -- Quais features o modelo mais usou

O XGBoost guarda, pra cada feature, o quanto ela pesou nas decisoes das arvores (`feature_importances_`). Isso ajuda a entender o que o modelo aprendeu -- por exemplo, se `retorno_lag1` pesa muito mais que `volatilidade_5d`, faz sentido, porque o retorno de ontem costuma ter mais relacao com o de amanha do que a volatilidade sozinha.

In [0]:
import pandas as pd

importancias = pd.DataFrame({
    "feature": features,
    "importancia": modelo.feature_importances_
}).sort_values("importancia", ascending=False)

display(importancias)

## Onde ver os resultados

No menu lateral do Databricks, clique em **Experiments** -- o run `xgboost_retorno_prox_dia` vai estar la, com os parametros, as metricas e o modelo salvo. Se voce rodar essa celula de novo com outro `max_depth` ou `n_estimators`, por exemplo, vai aparecer um novo run do lado, e da pra comparar os dois lado a lado.

Isso fecha a parte de ciencia de dados do projeto: fomos de dado bruto (Bronze) ate um modelo treinado e rastreado (MLflow), passando pela limpeza (Silver), pelas analises (Gold) e pelas features (notebook 05).